In [47]:
from tqdm import tqdm 

import pandas as pd
import requests

In [48]:
ORGANISM = 'Aedes'
RETMODE = 'json'
RETMAX = 100_000
TERM = f'("{ORGANISM}"[Organism] OR {ORGANISM}[All Fields]) AND "{ORGANISM}"[orgn]'


In [49]:
# Realizando requisição para obter os IDs relacionados ao termo de busca
def id_search_NCBI(params: dict) -> dict:

    esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    response = requests.get(esearch_url, params)
    response.raise_for_status() 

    data = response.json().get('esearchresult')

    return {
        'count': int(data.get('count')),
        'ids': data.get('idlist')
    }

ids_data = id_search_NCBI({
    'term': TERM,
    'retmax': RETMAX,
    'retmode': RETMODE,
    'db': 'sra',
})

ids = ids_data['ids']
print(f"IDs encontrados: {ids_data['count']}")
print(f"IDs obtidos: {len(ids)}")

IDs encontrados: 29965
IDs obtidos: 29965


In [50]:
def summary_NCBI(params: dict) -> dict:

    esummary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

    response = requests.get(esummary_url, params)
    response.raise_for_status() 

    data = response.json()
    return data

# Primeiramente devemos converter a lista de IDs em uma string de IDs 
list_ids = ','.join(ids[:100])          # Limitando aos 100 primeiros
print(f'Lista de IDs em string: {list_ids}')

# Busca do sumário via IDs do banco SRA 
sra_summary = summary_NCBI({
    'retmode': RETMODE, # JSON
    'db': 'sra',        # Banco de dados SRA
    'id': list_ids      # Lista de IDs em string
})

# Verificando metadados para o primeiro ID da lista
id = ids[0]
print('Metadados via SRA', sra_summary.get('result').get(id))

Lista de IDs em string: 42083430,42083429,42083428,42083427,42083426,42083425,42083424,42083423,42083422,42083421,42083420,42083419,42083418,42083417,42083416,42083415,42083414,42083413,42083412,42083411,42083410,42083409,42083408,42083407,42083406,42083405,42083404,42083403,42083402,42083401,42083400,42083399,42083398,42083397,42083396,42083395,42083394,42083393,42083392,42083391,42083390,42083389,44850497,44850496,44850495,44850494,44850493,44850492,44850491,44850490,44850489,44850488,44850487,44850486,44850485,44850484,44850483,44850482,44850481,44850480,44850479,44850478,44850477,44850476,44850475,44850474,44850473,44850472,44850471,44850470,44850469,44850468,44850467,44850466,44850465,44850464,44850463,44850462,44850461,44850460,44850459,44850458,44850457,44850456,44850455,44850454,44850453,44850452,44850451,44850450,44850449,44850448,44850447,44850446,44850445,44850444,44850443,44850442,44850441,44850440
Metadados via SRA {'uid': '42083430', 'expxml': '  <Summary><Title>mosquitoR

In [51]:
import xml.etree.ElementTree as et

def expxml_parse(xml_string: str) -> dict:

    xml = f'<SRA>{xml_string}</SRA>'
    root = et.fromstring(xml)

    # Armazenar dados 
    data = {}

    # Extraindo dados da primeira tag
    summary_tag = root.find('Summary')
    for child in summary_tag: 

        if not child.text == None: data[child.tag] = child.text 
        if child.attrib: 

            for key in child.attrib.keys(): data[f'{child.tag}_{key}'] = child.attrib[key]

    summary_tag = root.find('Library_descriptor')
    for child in summary_tag: 

        if not child.text == None: data[child.tag] = child.text 
        
        if child.attrib: 

            for key in child.attrib.keys(): data[f'{child.tag}_{key}'] = child.attrib[key]

    for child in root: 
        
        if not child.text == None: data[child.tag] = child.text 

        if child.attrib: 

            for key in child.attrib.keys(): data[f'{child.tag}_{key}'] = child.attrib[key]    
    
    return data

def runs_parse(xml_string: str) -> dict:

    xml = f'<SRA>{xml_string}</SRA>'
    root = et.fromstring(xml)

    data = {}

    for child in root: 

        if not child.text == None: data[f'{child.tag}'] = child.text 

        if child.attrib: 

            for key in child.attrib.keys(): data[f'{child.tag}_{key}'] = child.attrib[key]    

    return data

# Obtendo sumário para o primeiro ID
id = ids[0]
sumario = sra_summary.get('result').get(id)
print('Sumário: ', sumario)

# Verificando códigos expxmlp_parse 
expxml_data = expxml_parse(sumario.get('expxml').strip())
print('Expxml: ', expxml_data)

# Verificando códigos runs_parse 
runs_data = runs_parse(sumario.get('runs').strip())
print('Runs: ', runs_data)

Sumário:  {'uid': '42083430', 'expxml': '  <Summary><Title>mosquitoRNAi42 (CRX2019186; CRA030398)</Title><Platform instrument_model="Illumina HiSeq 4000">ILLUMINA</Platform><Statistics total_runs="1" total_spots="25113835" total_bases="7487355449" total_size="2748713976" load_done="true" cluster_name="public"/></Summary><Submitter acc="DRA025421" center_name="" contact_name="" lab_name=""/><Experiment acc="DRX848093" ver="1" status="public" name="mosquitoRNAi42 (CRX2019186; CRA030398)"/><Study acc="DRP016711" name="Reactive oxygen species-producing genes regulate mosquito midgut bacteria colonization, transcriptomic changes and cell repair (PRJCA046420)"/><Organism taxid="7159" ScientificName="Aedes aegypti"/><Sample acc="DRS627956" name=""/><Instrument ILLUMINA="Illumina HiSeq 4000"/><Library_descriptor><LIBRARY_NAME>missing</LIBRARY_NAME><LIBRARY_STRATEGY>RNA-Seq</LIBRARY_STRATEGY><LIBRARY_SOURCE>TRANSCRIPTOMIC</LIBRARY_SOURCE><LIBRARY_SELECTION>RANDOM</LIBRARY_SELECTION><LIBRARY_LAY

In [52]:
# Código para obtenção dos metadados completos
BATCH_SIZE = 200

metadata_list = []
for i in tqdm(range(0, len(ids), BATCH_SIZE)):
    
    batch = ids[i: i + BATCH_SIZE]

    sra_summary = summary_NCBI({
        'retmode': RETMODE,
        'db': 'sra',
        'id': ','.join(batch)
    })

    for id in batch:

        expxml = sra_summary.get('result').get(id).get('expxml').strip()
        runs = sra_summary.get('result').get(id).get('runs').strip()

        description_data = {
            'id': sra_summary.get('result').get(id).get('uid'),
            'createdate': sra_summary.get('result').get(id).get('createdate'),
            'updatedate': sra_summary.get('result').get(id).get('updatedate'),
        }

        expxml_data = expxml_parse(expxml)
        run_data = runs_parse(runs)

        data = description_data | run_data | expxml_data
        metadata_list.append(data)

df = pd.DataFrame(metadata_list)
df.head()

100%|██████████| 150/150 [01:44<00:00,  1.44it/s]


,id,createdate,updatedate,Run_acc,Run_total_spots,Run_total_bases,Run_load_done,Run_is_public,Run_cluster_name,Run_static_data_available,...,Instrument_BGISEQ,Instrument_DNBSEQ,Organism_CommonName,Instrument_PACBIO_SMRT,Instrument_ELEMENT,Instrument_CAPILLARY,Instrument_ION_TORRENT,Instrument_LS454,Instrument_ABI_SOLID,Statistics_static_data_available
0,42083430,2026/06/01,2025/12/16,DRR870143,25113835,7487355449,true,true,public,true,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,42083429,2026/06/01,2025/12/16,DRR870142,24795456,7380452801,true,true,public,true,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,42083428,2026/06/01,2025/12/16,DRR870141,22395959,6673387621,true,true,public,true,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,42083427,2026/06/01,2025/12/16,DRR870140,24092864,7183008156,true,true,public,true,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42083426,2026/06/01,2025/12/16,DRR870139,23805015,7099766031,true,true,public,true,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
import os

diretorio = f'../Data/{ORGANISM}'

os.makedirs(f'{diretorio}', exist_ok = True)
df.to_csv(f'{diretorio}/dados_brutos.csv', index = False) 